# Simple RL model: q-learning with 5-rounds of combat

Code sources: 
* https://www.tensorflow.org/agents/tutorials/intro_bandit
* 

In [2]:
#Import libraries 
import pandas as pd
import numpy as np
import requests
import matplotlib as plt
import seaborn as sns
import matplotlib.pyplot as plt


import requests
from bs4 import BeautifulSoup
import pandas as pd
from io import StringIO
import re
import random

import warnings
from pandas.errors import SettingWithCopyWarning
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=SettingWithCopyWarning)

### import datasets: 1 character and 1 monster

In [14]:
def open_clean_actionlist(filepath):
    actionlist = pd.read_csv(filepath)
    #clean up nan calues 
    actionlist= actionlist.replace('x', 0).fillna(0)
    return actionlist

char_actlist = open_clean_actionlist('../build data/rogue_actionlist.csv')
char_actlist.sample(3)

,Cost,requirement,action,action type,trait,Description,HP bonus,save effect,AC effect,target ac,...,attack roll,range,number die,damage die,additional damage,damage type,duration (round),cool down (round),crit success,crit failure
0,1,0,move towards target,movement,speed,25ft (5 squares),0,0,0,0,...,0,25,0,0,0,0,0,0,0.0,0.0
10,1,third attack,Dagger attack (third),melee,dexterity,0,0,0,0,0,...,1,0,1.00,4,3,P,0,0,0.0,0.0
5,1,first attack,rapier attack(first),melee,dexterity,0,0,0,0,0,...,7,0,1,6,3,P,0,0,0.0,0.0


In [15]:
monster_actlist = open_clean_actionlist('../build data/kobold scout_actionlist.csv')
monster_actlist.sample(3)

,Cost,requirement,action,action type,trait,Description,HP bonus,save effect,AC effect,target ac,...,attack roll,range,number die,damage die,additional damage,damage type,duration (round),cool down (round),crit success,crit failure
10,1,second attack,reload crossbow,range,strength,reload crossbow between attacks,0,0,0,0,...,0,0,0,0,0,0,0,0,0.0,0.0
13,1,target off-guard,sneak attack shortsword,melee,strength,deals more damage to off-guard target,0,0,0,0,...,9,0,1,12,0,0,0,0,0.0,0.0
7,1,third attack,shortsword (third),melee,strength,0,0,0,0,0,...,1,0,1,6,0,P,0,0,0.0,0.0


In [414]:
## import character build basic stats
build_skill = pd.read_csv('../build data/all_build_base_skills.csv')
char_skill = build_skill[build_skill['Unnamed: 0']== 'halfling rogue dexterity']
#rename columns to match monster format 
char_skill= char_skill.rename(columns={'hit points':'HP', 'armor class':'AC', 'Unnamed: 0':'character build'})
#make into a dictionary for eaiser analysis
char_skill_dict = char_skill.iloc[0].to_dict()
char_skill_dict

{'character build': 'halfling rogue dexterity',
 'ancestry': 'halfling',
 'class': 'rogue dexterity',
 'level': 1,
 'HP': 15.0,
 'speed': 25.0,
 'perception': 6.0,
 'fortitude': 3.0,
 'reflex': 7.0,
 'will': 6.0,
 'AC': 14.0,
 'unarmored': 2.0,
 'light': 2.0,
 'medium': 0.0,
 'heavy': 0.0,
 'unarmed': 2.0,
 'simple': 2.0,
 'martial': 0.0,
 'advanced': 0.0,
 'other': 0.0,
 'magical tradition': 0.0,
 'spell attack': 0.0,
 'spell dc': 10.0}

In [415]:
monster_build = pd.read_csv('../build data/monster_data_transform.csv', header=1)
monster_build.head(3)

#Check all monster options to make sure monster of interest is in the list 
monster_list = monster_build['name'].astype('str').unique().tolist()
#make it all lower case 
monster_list = [x.lower() for x in monster_list]
monster_check = 'kobold scout'
monster_name = [x for x in monster_list if re.search(monster_check, x)]
monster_name

#pull out the row for that monster stat 
choice_monster_build = monster_build[monster_build['name'].str.lower() == monster_name[0]]
#make all build options to a dictionary for easier analysis
mon_stat_dict= choice_monster_build.iloc[0].to_dict()

## establish encounter dynamics: 
* tracker for number of rounds
* Status update on character and monster health
* Decide on action options for each unique round
* calculate action result

In [416]:
### first round: each character rolls for initative 

#character initative= dice roll + perception 
char_intve = random.randint(1, 20) + char_sill_dict['perception'] 
print("character initative: ", char_intve)
mon_intve = random.randint(1, 20) + mon_stat_dict['Perception'] 
print("monster initative: ", mon_intve)

character initative:  17.0
monster initative:  13.0


In [395]:
# ## simulate first round 

# current_turn_player = 'character'
# combat_round = 1
# #Create a list to save the results 
# enc_result=[]
# #Define the target 
# target_ac= int(float(mon_stat_dict['AC']))
# target_health= int(float(mon_stat_dict['HP']))

# #Start the counter for the number of actions available
# action_num= 3
# while action_num >0:
    
#     ### Selection action from avaiable actions ###
    
#     if action_num == 3:
#         #pull out all options for first attack
#         actlist_filter = char_actlist[(char_actlist['requirement'] == 'first attack') | (char_actlist['requirement'] == 0)]
#         #filter out only attacks for first modeling 
#         attack_action= actlist_filter[(actlist_filter['action type'] == 'melee') | (actlist_filter['action type'] == 'range')]
#     elif action_num == 2:
#         #pull out all options for first attack
#         actlist_filter = char_actlist[(char_actlist['requirement'] == 'second attack') | (char_actlist['requirement'] == 0)]
#         #filter out only attacks for first modeling 
#         attack_action= actlist_filter[(actlist_filter['action type'] == 'melee') | (actlist_filter['action type'] == 'range')]
#     if action_num == 1:
#         #pull out all options for first attack
#         actlist_filter = char_actlist[(char_actlist['requirement'] == 'third attack') | (char_actlist['requirement'] == 0)]
#         #filter out only attacks for first modeling 
#         attack_action= actlist_filter[(actlist_filter['action type'] == 'melee') | (actlist_filter['action type'] == 'range')]
#     #pick a random action from the list 
#     attack_choice = attack_action.sample(n=1).iloc[0]
#     attack_choice
    
#     #-----#
#     ### Simulate the results of that action
#     #calculate the chance of hitting 
#     roll_calc = int(attack_choice['attack roll']) + random.randint(1, 20)
#     #roll_calc = 30
#     crit_level = max(int(attack_choice['attack roll']) + 20, int(target_ac) + 10)
#     crit_fail_level = int(attack_choice['attack roll']) + 1
    
#     #View for troubleshooting
#     #print(f'roll {roll_calc}, target AC {target_ac}, crit level {crit_level}, fail level {crit_fail_level}')
    
#     #results for a normal hit
#     if roll_calc >= target_ac and roll_calc <= crit_level:
#         #print("sucess") #calculate total damage 
#         die_max = int(attack_choice['damage die'])
#         num_dmg_die= int(float(attack_choice['number die']))
#         total_damage = (num_dmg_die * random.randint(1, die_max)) + int(attack_choice['additional damage'])
#         attack_sucess = 1
#     #results for a critical hit 
#     elif roll_calc >=target_ac and roll_calc >= crit_level:
#        # print("crit sucess") #calculate total damage 
#         die_max = int(attack_choice['damage die'])
#         num_dmg_die= int(float(attack_choice['number die']))
#         total_damage = (num_dmg_die * random.randint(1, die_max)) + (num_dmg_die * random.randint(1, die_max))+ int(attack_choice['additional damage'])
#         attack_sucess = 2
#     #results for a critical fail 
#     elif roll_calc < target_ac and roll_calc <= crit_fail_level:
#         #print("crit fail") #calculate total damage 
#         total_damage = 0
#         attack_sucess = -1
#     #results for a normal fail
#     elif roll_calc < target_ac:
#         #print('fail')
#         total_damage = 0
#         attack_sucess = 0
#         #print('total damage =', total_damage)
#     #store read errors as failures
#     else:
#         #print('other')
#         total_damage = 0
#         attack_sucess = 0
#         #print('total damage =', total_damage)
    
#     ######calculate the 'reward' or change in state based on these results #####
    
#     #Current health or AC changes based on the results 
#     target_health = target_health - total_damage

#     #calculate action reward
#     attack_reward= attack_sucess*total_damage
#     if target_health <=0:
#         attack_reward= attack_reward + 100
    
#     #Save the results in the results_list 
#     enc_result.append([current_turn_player, combat_round, action_num, attack_choice['action'], attack_sucess, total_damage, target_health, attack_reward])
    
#     #Calculate the action cost 
#     action_num = action_num - attack_choice['Cost']

# #View result of choices 
# results_col= ['player', 'combat_round', 'action_num', 'attack_choice', 'attack_sucess', 'total_damage', 'mon_health', 'attack_reward']

# results_df= pd.DataFrame(enc_result, columns= results_col)
# results_df

,player,combat_round,action_num,attack_choice,attack_sucess,total_damage,mon_health,attack_reward
0,character,1,3,Dagger attack (first),1,5,11,5
1,character,1,2,rapier attack(second),0,0,11,0
2,character,1,1,Throw Dagger attack (third),0,0,11,0


In [454]:
def round_physical_attack_result(current_player, char_actlist, combat_round, target_health, target_ac):
    #mon_stat_dict= target_dict
    current_turn_player = current_player
    #Define the target 
    target_ac= target_ac
    target_health = target_health
    
    #Create a list to save the results 
    enc_result=[]
    #Start the counter for the number of actions available
    action_num= 3
    while action_num >0:
        
        ### Selection action from avaiable actions ###
        
        if action_num == 3:
            #pull out all options for first attack
            actlist_filter = char_actlist[(char_actlist['requirement'] == 'first attack') | (char_actlist['requirement'] == 0)]
            #filter out only attacks for first modeling 
            attack_action= actlist_filter[(actlist_filter['action type'] == 'melee') | (actlist_filter['action type'] == 'range')]
        elif action_num == 2:
            #pull out all options for first attack
            actlist_filter = char_actlist[(char_actlist['requirement'] == 'second attack') | (char_actlist['requirement'] == 0)]
            #filter out only attacks for first modeling 
            attack_action= actlist_filter[(actlist_filter['action type'] == 'melee') | (actlist_filter['action type'] == 'range')]
        if action_num == 1:
            #pull out all options for first attack
            actlist_filter = char_actlist[(char_actlist['requirement'] == 'third attack') | (char_actlist['requirement'] == 0)]
            #filter out only attacks for first modeling 
            attack_action= actlist_filter[(actlist_filter['action type'] == 'melee') | (actlist_filter['action type'] == 'range')]
        #pick a random action from the list 
        attack_choice = attack_action.sample(n=1).iloc[0]
        attack_choice
        
        #-----#
        ### Simulate the results of that action
        #calculate the chance of hitting 
        roll_calc = int(attack_choice['attack roll']) + random.randint(1, 20)
        #roll_calc = 30
        crit_level = max(int(attack_choice['attack roll']) + 20, int(target_ac) + 10)
        crit_fail_level = int(attack_choice['attack roll']) + 1
        
        #View for troubleshooting
        #print(f'roll {roll_calc}, target AC {target_ac}, crit level {crit_level}, fail level {crit_fail_level}')
        
        #results for a normal hit
        if roll_calc >= target_ac and roll_calc <= crit_level:
            #print("sucess") #calculate total damage 
            die_max = int(attack_choice['damage die'])
            if die_max ==0: 
                die_max = 4
            num_dmg_die= int(float(attack_choice['number die']))
            total_damage = (num_dmg_die * random.randint(1, die_max)) + int(attack_choice['additional damage'])
            attack_sucess = 1
        #results for a critical hit 
        elif roll_calc >=target_ac and roll_calc >= crit_level:
           # print("crit sucess") #calculate total damage 
            die_max = int(attack_choice['damage die'])
            if die_max ==0: 
                die_max = 4
            num_dmg_die= int(float(attack_choice['number die']))
            total_damage = (num_dmg_die * random.randint(1, die_max)) + (num_dmg_die * random.randint(1, die_max))+ int(attack_choice['additional damage'])
            attack_sucess = 2
        #results for a critical fail 
        elif roll_calc < target_ac and roll_calc <= crit_fail_level:
            #print("crit fail") #calculate total damage 
            total_damage = 0
            attack_sucess = -1
        #results for a normal fail
        elif roll_calc < target_ac:
            #print('fail')
            total_damage = 0
            attack_sucess = 0
            #print('total damage =', total_damage)
        #store read errors as failures
        else:
            #print('other')
            total_damage = 0
            attack_sucess = 0
            #print('total damage =', total_damage)
        
        ######calculate the 'reward' or change in state based on these results #####
        
        #Current health or AC changes based on the results 
        target_health = target_health - total_damage
    
        #calculate action reward
        attack_reward= attack_sucess*total_damage
        if target_health <=0:
            attack_reward= attack_reward + 100
        
        #Save the results in the results_list 
        enc_result.append([current_turn_player, combat_round, action_num, attack_choice['action'], attack_sucess, total_damage, target_health, attack_reward])
        
        #Calculate the action cost 
        action_num = action_num - attack_choice['Cost']
    
    #View result of choices 
    results_col= ['player', 'combat_round', 'action_num', 'attack_choice', 'attack_sucess', 'total_damage', 'target_health', 'attack_reward']
    
    results_df= pd.DataFrame(enc_result, columns= results_col)
    return results_df, target_health

# Simulate 5 rounds of player combat

In [427]:
# ### first round: each character rolls for initative 

# #character initative= dice roll + perception 
# char_intve = random.randint(1, 20) + char_sill_dict['perception'] 
# print("character initative: ", char_intve)
# mon_intve = random.randint(1, 20) + mon_stat_dict['Perception'] 
# print("monster initative: ", mon_intve)

# #define starting health and AC values
# target_ac= int(float(mon_stat_dict['AC']))
# target_health= int(float(mon_stat_dict['HP']))

# #Save results to a main dataframe 
# full_enc_results= pd.DataFrame()

# #Simulate 5 rounds of combat 
# char_round_1, target_health = round_physical_attack_result('rogue', char_actlist, 1, target_health, target_ac)
# full_enc_results= pd.concat([full_enc_results, char_round_1])
# print(f'round 1, {target_health}')

# char_round_2, target_health = round_physical_attack_result('rogue', char_actlist, 2, target_health, target_ac)
# full_enc_results= pd.concat([full_enc_results, char_round_2])
# print(f'round 2, {target_health}')

# char_round_3, target_health = round_physical_attack_result('rogue', char_actlist, 3, target_health, target_ac)
# full_enc_results= pd.concat([full_enc_results, char_round_3])
# print(f'round 3, {target_health}')

# char_round_4, target_health = round_physical_attack_result('rogue', char_actlist, 4, target_health, target_ac)
# full_enc_results= pd.concat([full_enc_results, char_round_4])
# print(f'round 4, {target_health}')

# char_round_5, target_health = round_physical_attack_result('rogue', char_actlist, 5, target_health, target_ac)
# full_enc_results= pd.concat([full_enc_results, char_round_5])
# print(f'round 5, {target_health}')

# full_enc_results

character initative:  10.0
monster initative:  19.0
round 1, 16.0
round 2, 16.0
round 3, 16.0
round 4, 16.0
round 5, 16.0


,player,combat_round,action_num,attack_choice,attack_sucess,total_damage,mon_health,attack_reward
0,rogue,1,3,rapier attack(first),0,0,16.0,0
1,rogue,1,2,Throw Dagger attack (second),1,7,9.0,7
2,rogue,1,1,rapier attack (third),0,0,9.0,0
0,rogue,2,3,Throw Dagger attack (first),0,0,16.0,0
1,rogue,2,2,rapier attack(second),1,6,10.0,6
2,rogue,2,1,Dagger attack (third),0,0,10.0,0
0,rogue,3,3,rapier attack(first),1,6,10.0,6
1,rogue,3,2,Dagger attack (second),1,6,4.0,6
2,rogue,3,1,Dagger attack (third),1,4,0.0,104
0,rogue,4,3,Dagger attack (first),1,5,11.0,5


### Simulate number of rounds until one of the two players die

In [455]:
#####establish the player starting stats
#character initative= dice roll + perception 
char_intve = random.randint(1, 20) + char_skill_dict['perception'] 
print("character initative: ", char_intve)
mon_intve = random.randint(1, 20) + mon_stat_dict['Perception'] 
print("monster initative: ", mon_intve)

#define starting health and AC values
mon_ac= int(float(mon_stat_dict['AC']))
mon_health= int(float(mon_stat_dict['HP']))
print(f'monster HP: {mon_health}, monster AC: {mon_ac}')

char_ac= int(float(char_skill_dict['AC']))
char_health= int(float(char_skill_dict['HP']))
print(f'character HP: {char_health}, character AC: {char_ac}')

#Save results to a main dataframe 
full_enc_results= pd.DataFrame()

# #troubleshooting 

# char_intve= 2
# mon_intve = 1

### Simulate the results from one round of combat

if char_intve >= mon_intve:
    round_number = 1
    while mon_health >0 or char_health >0:
        char_round, mon_health = round_physical_attack_result('rogue', char_actlist, round_number, mon_health, mon_ac)
        full_enc_results= pd.concat([full_enc_results, char_round])
        print(f'{round_number}: monster HP {mon_health}')

        if mon_health < 0: 
            break
    
        mon_round, char_health = round_physical_attack_result('kobold', monster_actlist, round_number, char_health, char_ac)
        full_enc_results= pd.concat([full_enc_results, mon_round])
        print(f'{round_number}: character HP {char_health}')

        if char_health < 0: 
            break

        round_number +=1

elif char_intve < mon_intve:
    round_number = 1
    while mon_health >0 or char_health >0:
        mon_round, char_health = round_physical_attack_result('kobold', monster_actlist, round_number, char_health, char_ac)
        full_enc_results= pd.concat([full_enc_results, mon_round])
        print(f'{round_number}: character HP {char_health}')

        if char_health < 0: 
            break

        char_round, mon_health = round_physical_attack_result('rogue', char_actlist, round_number, mon_health, mon_ac)
        full_enc_results= pd.concat([full_enc_results, char_round])
        print(f'{round_number}: monster HP {mon_health}')

        if mon_health < 0: 
            break
        round_number +=1

full_enc_results

character initative:  15.0
monster initative:  14.0
monster HP: 16, monster AC: 18
character HP: 15, character AC: 14
1: monster HP 7
1: character HP 13
2: monster HP 1
2: character HP 13
3: monster HP -3


,player,combat_round,action_num,attack_choice,attack_sucess,total_damage,target_health,attack_reward
0,rogue,1,3,rapier attack(first),1,9,7,9
1,rogue,1,2,Throw Dagger attack (second),0,0,7,0
2,rogue,1,1,rapier attack (third),0,0,7,0
0,kobold,1,3,crossbow (first),0,0,15,0
1,kobold,1,2,shortsword (second),0,0,15,0
2,kobold,1,1,shortsword (third),1,2,13,2
0,rogue,2,3,rapier attack(first),1,6,1,6
1,rogue,2,2,rapier attack(second),0,0,1,0
2,rogue,2,1,rapier attack (third),0,0,1,0
0,kobold,2,3,crossbow (first),0,0,13,0
